In [1]:
import os
os.environ['ADASP_DATA'] = '/tsi'
from adasp_data_management import music, event, speaker

/home/ids/xzhuang/anaconda3/envs/adasp_data_management/lib/python3.10/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [5]:
db = event.BirdClef()

BirdClef: load metadata file /tsi/dcase/BirdClef/ttahara_birdsong-resampled-train-audio-00/metadata.csv


In [6]:
db.pdf_metadata

,file_path,rating,playback_used,ebird_code,date,pitch,filename,speed,species,number_of_notes,...,length,time,recordist,license,resampled_sampling_rate,resampled_channels,n_channels,sampling_rate_y,duration,cutoff_freq
0,/tsi/dcase/BirdClef/ttahara_birdsong-resampled...,3.5,no,aldfly,2013-05-25,Not specified,XC134874.mp3,Not specified,Alder Flycatcher,Not specified,...,Not specified,8:00,Jonathon Jongsma,Creative Commons Attribution-ShareAlike 3.0,32000,1 (mono),1,32000,25.488,9468
1,/tsi/dcase/BirdClef/ttahara_birdsong-resampled...,4.0,no,aldfly,2013-05-27,both,XC135454.mp3,both,Alder Flycatcher,1-3,...,0-3(s),08:30,Mike Nelson,Creative Commons Attribution-NonCommercial-Sha...,32000,1 (mono),1,32000,36.31021875,11078
2,/tsi/dcase/BirdClef/ttahara_birdsong-resampled...,4.0,no,aldfly,2013-05-27,both,XC135455.mp3,both,Alder Flycatcher,1-3,...,0-3(s),08:30,Mike Nelson,Creative Commons Attribution-NonCommercial-Sha...,32000,1 (mono),1,32000,39.2359375,10328
3,/tsi/dcase/BirdClef/ttahara_birdsong-resampled...,3.5,no,aldfly,2013-05-27,both,XC135456.mp3,both,Alder Flycatcher,1-3,...,0-3(s),08:30,Mike Nelson,Creative Commons Attribution-NonCommercial-Sha...,32000,1 (mono),1,32000,33.5935,10812
4,/tsi/dcase/BirdClef/ttahara_birdsong-resampled...,4.0,no,aldfly,2013-05-27,both,XC135457.mp3,level,Alder Flycatcher,1-3,...,0-3(s),08:30,Mike Nelson,Creative Commons Attribution-NonCommercial-Sha...,32000,1 (mono),1,32000,36.022875,11515
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21370,/tsi/dcase/BirdClef/ttahara_birdsong-resampled...,4.5,no,yetvir,2019-05-15,both,XC477608.mp3,level,Yellow-throated Vireo,4-6,...,0-3(s),13:00,Sue Riffe,Creative Commons Attribution-NonCommercial-Sha...,32000,1 (mono),,,,
21371,/tsi/dcase/BirdClef/ttahara_birdsong-resampled...,3.5,no,yetvir,2017-05-14,Not specified,XC500348.mp3,Not specified,Yellow-throated Vireo,Not specified,...,Not specified,15:00,Jacob Saucier,Creative Commons Attribution-NonCommercial-Sha...,32000,1 (mono),,,,
21372,/tsi/dcase/BirdClef/ttahara_birdsong-resampled...,5.0,no,yetvir,2017-06-10,Not specified,XC501230.mp3,Not specified,Yellow-throated Vireo,Not specified,...,Not specified,13:30,Jacob Saucier,Creative Commons Attribution-NonCommercial-Sha...,32000,1 (mono),,,,
21373,/tsi/dcase/BirdClef/ttahara_birdsong-resampled...,3.5,no,yetvir,2009-05-06,level,XC54828.mp3,level,Yellow-throated Vireo,4-6,...,>10(s),9:45am,Mike Nelson,Creative Commons Attribution-NonCommercial-Sha...,32000,1 (mono),,,,


In [2]:
import os

def create_mirror_structure(metadata_df, mirror_dir):
    # Ensure mirror_dir is absolute
    mirror_dir = os.path.abspath(mirror_dir)
    os.makedirs(mirror_dir, exist_ok=True)
    
    for _, row in metadata_df.iterrows():
        # Get the path from your metadata
        raw_src = row['file_path']
        
        # Use realpath to get the 'true' physical path
        src_path = os.path.realpath(raw_src)
        
        # Double check visibility in Python
        if not os.path.exists(src_path):
            print(f"Python cannot see: {src_path}")
            continue

        label = row['id']
        target_dir = os.path.join(mirror_dir, label)
        os.makedirs(target_dir, exist_ok=True)
        
        # Define target file path
        target_file = os.path.join(target_dir, os.path.basename(src_path))
        
        # Clear existing symlinks to avoid 'File exists' errors
        if os.path.lexists(target_file):
            os.remove(target_file)
            
        # Create the link using the fully resolved source path
        os.symlink(src_path, target_file)

In [3]:
vox = speaker.VoxCeleb1()
vox_metadata = vox.pdf_metadata
vox_metadata.iloc[0]

VoxCeleb1: load metadata file /tsi/plato_sons/speech/VoxCeleb/VoxCeleb1/metadata.csv


file_path        /tsi/plato_sons/speech/VoxCeleb/VoxCeleb1/vox1...
id                                                         id10053
VGG_face1_id                                      Andrew_Lee_Potts
gender                                                           m
nationality                                                     UK
set                                                            dev
n_speakers                                                       1
n_channels                                                       1
sampling_rate                                                16000
duration                                                 12.400063
cutoff_freq                                                   5906
Name: 0, dtype: object

In [4]:

# --- WORKFLOW ---
# 1. Get your metadata from your private package
# metadata = your_private_package.get_metadata()

# 2. Setup the Mirror
# MIRROR_AUDIO_PATH = 'data/BirdClef_Audio_Mirror'

MIRROR_AUDIO_PATH = 'Datasets/VoxCeleb1_Mirror'
create_mirror_structure(vox_metadata, MIRROR_AUDIO_PATH)

In [16]:
vox_metadata['id'].nunique()

1251

In [10]:
# 1. Get the frequency of each speaker ID
counts = vox_metadata['id'].value_counts()

# 2. Filter for IDs where the count is less than 10
low_sample_classes = counts[counts < 10].index.tolist()

print(f"Number of classes with < 10 samples: {len(low_sample_classes)}")
print("First few classes to remove:", low_sample_classes[:5])

Number of classes with < 10 samples: 0
First few classes to remove: []
